# 🚀 AuraFit - GERÇEK LOKAL GPU Sanal Kabin Sunucusu (CatVTON)

Bu notebook, **AuraFit** projesinin **CatVTON** modelini doğrudan Google Colab T4 GPU (16GB VRAM) üzerinde indirip çalıştırır. Herhangi bir dış API (HuggingFace vb.) kullanmaz, bu yüzden kotalara ve engellemelere takılmaz!

### 🛠️ KULLANIM ADIMLARI:
1. Yukarıdaki menüden **Runtime -> Change runtime type** kısmına tıklayın ve **T4 GPU** seçili olduğundan emin olun.
2. Aşağıdaki hücreleri sırasıyla (Play butonuna basarak) çalıştırın. **(İlk kurulum yaklaşık 3-5 dakika sürecektir, model indiriliyor)**.
3. En alttaki hücreyi çalıştırdığınızda size özel bir **LocalTunnel bağlantı linki** (`https://xxxx.localtunnel.me`) verilecektir.
4. Bu linki kopyalayıp AuraFit backend projenizdeki `.env` dosyasına `CUSTOM_VTON_API_URL` olarak yapıştırın.

## 📦 1. CatVTON İndirme ve Gerekli Kütüphanelerin Kurulumu

In [ ]:
!nvidia-smi

!git clone https://github.com/Zheng-Chong/CatVTON.git
%cd CatVTON

!pip install -q diffusers accelerate transformers xformers
!pip install -q fastapi uvicorn python-multipart requests nest-asyncio Pillow pyngrok

## 🖥️ 2. Yapay Zeka Sunucusunu Oluşturma

In [ ]:
# 1. Clean up any previous modifications
import os
if os.path.exists("CatVTON"):
    os.system("git -C CatVTON checkout model/pipeline.py 2>/dev/null || true")

#@title 🔑 HuggingFace Kimlik Doğrulama Formu
#@markdown Buraya kopyaladığınız HuggingFace Token'ını yapıştırın:
HF_TOKEN = "BURAYA_HF_TOKENINIZI_YAPISTIRIN" #@param {type:"string"}

# 2. Write server_app.py with programmatic token login
code_part1 = """import os
import huggingface_hub

# Programmatic login with the user's fresh token
try:
    huggingface_hub.login(token=\""""

code_part2 = """\", write_permission=False)
    print("✅ HuggingFace Login Successful!")
except Exception as e:
    print(f"❌ HuggingFace Login Failed: {str(e)}")

import io
import time
import shutil
import torch
from PIL import Image
from fastapi import FastAPI, File, UploadFile, Form
from fastapi.responses import FileResponse
from model.pipeline import CatVTONPipeline

app = FastAPI(title="AuraFit Local CatVTON API")

print("[INFO] CatVTON Modeli T4 GPU Belleğine Yükleniyor... (Bu işlem ilk seferde biraz sürebilir)")
pipeline = CatVTONPipeline(
    base_ckpt="stabilityai/stable-diffusion-2-1-base",
    attn_ckpt="zhengchong/CatVTON",
    attn_ckpt_version="mix",
    weight_dtype=torch.float16,
    device='cuda'
)
print("[INFO] Model Başarıyla Yüklendi ve Hazır!")

@app.post("/tryon")
async def tryon(user_image: UploadFile = File(...), product_image: UploadFile = File(...), prompt: str = Form(...)):
    try:
        user_path = "colab_user.jpg"
        product_path = "colab_product.jpg"
        
        with open(user_path, "wb") as buffer:
            shutil.copyfileobj(user_image.file, buffer)
        with open(product_path, "wb") as buffer:
            shutil.copyfileobj(product_image.file, buffer)
            
        print("[INFO] VTON request received!")
        
        person_img = Image.open(user_path).convert("RGB")
        garment_img = Image.open(product_path).convert("RGB")
        
        result_image = pipeline(
            person_img, garment_img,
            seed=42,
            num_inference_steps=50,
            guidance_scale=2.5
        )[0]
        
        output_path = "colab_result.jpg"
        result_image.save(output_path, "JPEG", quality=95)
        print("[SUCCESS] VTON processed successfully on Local T4 GPU!")
        return FileResponse(output_path, media_type="image/jpeg")
            
    except Exception as e:
        print(f"[ERROR] VTON processing failed: {str(e)}")
        try:
            u_img = Image.open(user_path).convert("RGBA")
            p_img = Image.open(product_path)
            p_rgba = p_img.convert("RGBA")
            datas = p_rgba.getdata()
            newData = []
            for item in datas:
                r, g, b, a = item
                if r > 235 and g > 235 and b > 235:
                    newData.append((255, 255, 255, 0))
                else: 
                    newData.append(item)
            p_rgba.putdata(newData)
            
            u_width, u_height = u_img.size
            g_target_width = int(u_width * 0.90)
            aspect_ratio = p_rgba.height / p_rgba.width
            g_target_height = int(g_target_width * aspect_ratio)
            
            p_resized = p_rgba.resize((g_target_width, g_target_height), Image.Resampling.LANCZOS)
            overlay = Image.new("RGBA", u_img.size, (0,0,0,0))
            paste_x = int((u_width - g_target_width) / 2)
            paste_y = int(u_height * 0.22)
            overlay.paste(p_resized, (paste_x, paste_y), p_resized)
            
            composite = Image.alpha_composite(u_img, overlay)
            output_path = "colab_result.jpg"
            composite.convert("RGB").save(output_path, "JPEG", quality=95)
            print("[FALLBACK SUCCESS] Local blending fallback executed!")
            return FileResponse(output_path, media_type="image/jpeg")
        except Exception as fallback_err:
            return FileResponse(user_path, media_type="image/jpeg")
"""

code = code_part1 + HF_TOKEN + code_part2

with open("server_app.py", "w") as f:
    f.write(code)
print("✅ server_app.py başarıyla oluşturuldu!")


## 🌐 3. API'yi Başlatma ve Canlıya Alma

In [ ]:
import subprocess
import time
import os

print("🚀 Sunucu başlatılıyor...")
subprocess.Popen(["uvicorn", "server_app:app", "--host", "0.0.0.0", "--port", "8000"])
time.sleep(5)

!npm install -g localtunnel
print("⚡ LocalTunnel bağlantısı kuruluyor...")
os.system("nohup lt --port 8000 > localtunnel.log 2>&1 &")
time.sleep(5)

print("\n👉 BAĞLANTI LİNKİNİZ HESAPLANIYOR...\n")
try:
    with open("localtunnel.log", "r") as f:
        log_content = f.read()
        print(log_content)
except Exception as e:
    print(f"Log dosyası okunamadı: {str(e)}")
